# Aqua API — Train & Predict demo

End-to-end walk-through of the typical client flow:

1. **Authenticate** with username + password.
2. **Create a Bible version** (a logical container for one or more revisions).
3. **Upload a revision** of your translation as a vref-aligned text file. *(In a future version this manual upload step will be replaced by a more convenient integration with the Paratext text-storage API, so the API can pull translations directly rather than asking you to upload them.)*
4. **Look up the reference revision** you want to align against (here: NIV, revision id `304`).
5. **Trigger training** for all available apps (semantic-similarity, tfidf, word-alignment, ngrams, agent-critique).
6. **Poll training status** until every job reaches a terminal state.
7. **Inspect training results** — interleaved per-vref scores from each app.
8. **Run `/predict`** on a new (or adjusted) verse pair using the artifacts you just trained.

Edit the `BASE_URL`, `USERNAME`, and `PASSWORD` cells to point at the environment you want to use.

> The Aqua API uses the [vref convention](https://github.com/BibleNLP/ebible/blob/main/metadata/vref.txt) — every revision is a 41,899-line text file where line N corresponds to canonical verse reference N. Lines may be blank for verses you haven't translated.

## What each app does

The training / predict fan-out covers five apps. Each scores a `(source_text, target_text)` pair on a different dimension; the design intent is that they're complementary rather than redundant.

| App | What it does | Strength |
| --- | --- | --- |
| **semantic-similarity** | Compares sentence-pair embeddings from a fine-tuned model. | Catches meaning drift even when the two sides share no surface vocabulary (e.g. a paraphrase that lost the actual meaning). |
| **tfidf** | Builds a [TF-IDF](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) + truncated-SVD encoder over the corpus and returns the nearest-neighbour verses to a query — i.e. the verses whose wording is most similar. | Useful when reviewing a verse: surfaces other places in the corpus that read similarly, so the translator can check consistency or pull comparable renderings. |
| **word-alignment** | Word-level alignment between source and target. With `use_eflomal=true` it uses [eflomal](https://github.com/robertostling/eflomal) (statistical alignment); otherwise a neural aligner. | Surfaces added words, omitted words, and systematic mistranslations of specific terms. |
| **ngrams** | Mines frequent multi-word phrases from the training corpus, then scores how well a pair's text overlaps with that learned ngram set. | Catches verses that are missing the recurring phrases your translation team usually uses. |
| **agent-critique** *(likely to be renamed `ai-notes`)* | LLM-driven pipeline. Produces structured per-verse issues (omissions, additions, replacements, severity), an automatic back-translation, lexeme cards, and a grammar sketch. | Surfaces semantic and theological issues a statistical model would miss, plus the back-translation / lexeme / grammar artefacts give human reviewers context to work from. Most expensive of the five — call it via `/predict` on the verses you actually want inspected, not over the whole training set. |

You can fan out to a subset by passing `apps=[...]` on `/train` or `/predict`. Omit it to run all five.

In [ ]:
import getpass
import json
import time
from pathlib import Path

import requests


def check(resp):
    """Like ``resp.raise_for_status()`` but surfaces the response body so a
    422 from Pydantic actually tells you which field it didn't like."""
    if not resp.ok:
        raise RuntimeError(
            f"{resp.request.method} {resp.url} -> "
            f"{resp.status_code}: {resp.text}"
        )
    return resp

## 0. Configure

In [ ]:
# Choose the environment you want to hit.
BASE_URL = "https://cp3by92k8p.us-east-1.awsapprunner.com"  # development
# BASE_URL = "https://tmv9bz5v4q.us-east-1.awsapprunner.com"  # production

# All v3 routes are reachable at /latest/ as well — we use /latest/ here.
PREFIX = "latest"

USERNAME = "your-username"  # e.g. your email or login name
PASSWORD = getpass.getpass("Password: ")  # entered interactively

# A reference revision to align against. 304 is the NIV in the dev/prod DBs.
REFERENCE_REVISION_ID = 304

# vref-aligned text file representing your translation. We pull a copy of the
# Nyanja (nya) Bible from BibleNLP's ebible corpus on GitHub. Nyanja
# (a.k.a. Chichewa) is a Bantu language spoken in Malawi, Zambia, and
# Mozambique. Replace the URL with your own translation when you're ready.
REVISION_URL = (
    "https://raw.githubusercontent.com/BibleNLP/ebible/main/corpus/nya-nya.txt"
)

## 1. Authenticate

`POST /latest/token` with form-encoded credentials returns a Bearer token. Every subsequent request includes that token in the `Authorization` header.

In [ ]:
resp = check(requests.post(
    f"{BASE_URL}/{PREFIX}/token",
    data={"username": USERNAME, "password": PASSWORD},
))
TOKEN = resp.json()["access_token"]
AUTH = {"Authorization": f"Bearer {TOKEN}"}
print(f"Got token (length: {len(TOKEN)})")

## 2. Create a Bible version

A *version* is a logical container. *Revisions* (specific translations) live under it. The same translation project at different stages can be uploaded as separate revisions of one version.

If you already have a version you want to add a revision to, skip this step and use its `id` directly.

In [ ]:
# Look up your groups so the new version can be shared with one of them.
# /version requires `add_to_groups` to be non-empty — without a group, no
# other user could see the version. Pick deliberately: this version will
# be visible to every member of every group you list here.
my_groups = check(requests.get(f"{BASE_URL}/{PREFIX}/groups/me", headers=AUTH)).json()
assert my_groups, "You don't belong to any groups; ask an admin to add you to one."

print(f"You belong to {len(my_groups)} group(s):")
for g in my_groups:
    print(f"  id={g['id']:<5}  name={g['name']}")

# Default: share only with your first group. Override `version_groups` to
# pick a different one, or pass several IDs if you really want to.
version_groups = [my_groups[0]["id"]]
print(f"\nThis version will be added to group(s): {version_groups}")

version_payload = {
    "name": "Demo Nyanja Version",
    "iso_language": "nya",  # ISO 639-3 — Nyanja / Chichewa
    "iso_script": "Latn",   # ISO 15924
    "abbreviation": "NYADEMO",
    "rights": "Public domain demo",
    "machineTranslation": False,
    "is_reference": False,
    "add_to_groups": version_groups,
}

resp = check(requests.post(
    f"{BASE_URL}/{PREFIX}/version",
    json=version_payload,
    headers=AUTH,
))
version = resp.json()
VERSION_ID = version["id"]
print(f"\nCreated version {VERSION_ID}: {version['name']!r}")
version

## 3. Upload a revision

`POST /latest/revision` accepts query params (`version_id`, `name`) and a multipart file upload of the vref-aligned text.

The server validates the file is exactly 41,899 lines (one per canonical vref) and stores it linked to the version.

In [ ]:
# Pull the revision text directly from the BibleNLP ebible corpus on GitHub
# and forward it as a multipart upload — no need to write a local file.
revision_bytes = check(requests.get(REVISION_URL)).content
line_count = revision_bytes.count(b"\n") + (0 if revision_bytes.endswith(b"\n") else 1)
print(f"Downloaded {len(revision_bytes):,} bytes ({line_count:,} lines)")

resp = check(requests.post(
    f"{BASE_URL}/{PREFIX}/revision",
    params={"version_id": VERSION_ID, "name": "Demo Revision"},
    files={"file": ("nya-nya.txt", revision_bytes)},
    headers=AUTH,
))
revision = resp.json()
REVISION_ID = revision["id"]
print(f"Uploaded revision {REVISION_ID}: {revision['name']!r}")

# Stash the lines so the predict cell below can pull a real verse out.
REVISION_LINES = revision_bytes.decode("utf-8").splitlines()
revision

## 4. Resolve the reference (NIV) version_id

Training is keyed on revisions; predict is keyed on **versions**. We need the NIV's `bible_version_id` for the predict step later.

If you already know it, you can hardcode `REFERENCE_VERSION_ID` directly. Otherwise, the lookup below scans your accessible revisions for `id == REFERENCE_REVISION_ID`.

In [ ]:
resp = check(requests.get(f"{BASE_URL}/{PREFIX}/revision", headers=AUTH))
revisions = resp.json()

ref = next((r for r in revisions if r["id"] == REFERENCE_REVISION_ID), None)
if ref is None:
    raise RuntimeError(
        f"Revision {REFERENCE_REVISION_ID} is not in your accessible-revisions list. "
        "Ask your admin to grant your group access to the NIV reference version, "
        "or set REFERENCE_VERSION_ID directly below."
    )
REFERENCE_VERSION_ID = ref["bible_version_id"]
print(
    f"Reference (revision {REFERENCE_REVISION_ID}) lives under "
    f"version_id {REFERENCE_VERSION_ID}"
)

## 5. Trigger training

`POST /latest/train` queues training jobs for every app at once and returns a `session_id` you can poll. Pass `apps=[...]` to restrict to specific apps; omit it to train all of them.

`use_eflomal=true` opts the word-alignment app into the eflomal alignment path.

In [ ]:
train_payload = {
    "source_revision_id": REVISION_ID,            # your translation
    "target_revision_id": REFERENCE_REVISION_ID,  # NIV reference
    "options": {"use_eflomal": True},             # Will probably become default soon
    # "apps": ["semantic-similarity", "word-alignment"]  # optional subset
}

resp = check(requests.post(
    f"{BASE_URL}/{PREFIX}/train",
    json=train_payload,
    headers=AUTH,
))
training_response = resp.json()
SESSION_ID = training_response["session_id"]
print(f"Session id: {SESSION_ID}")
for job in training_response["training_jobs"]:
    print(f"  job {job['id']:>4}  {job['type']:<22}  status={job['status']}")

## 6. Poll training status

`GET /latest/train/status?session_id=...` returns the same payload shape as `/train`, so you can watch each job tick from `queued` → `running` → `finished` (or `failed`).

Typical durations on dev with the Nyanja Bible demo revision:

| App | Typical |
| --- | --- |
| semantic-similarity | ~1 min |
| tfidf | ~1–2 min |
| ngrams | ~3 min |
| word-alignment (eflomal) | ~10–18 min |
| agent-critique | ~1–9 min (variable) |

In [ ]:
TERMINAL = {"finished", "failed", "cancelled"}
POLL_INTERVAL = 30  # seconds

while True:
    resp = check(requests.get(
        f"{BASE_URL}/{PREFIX}/train/status",
        params={"session_id": SESSION_ID},
        headers=AUTH,
    ))
    jobs = resp.json()["training_jobs"]

    print(f"[{time.strftime('%H:%M:%S')}]")
    for job in jobs:
        pct = (
            f"{job['percent_complete']:>5.1f}%"
            if job.get("percent_complete") is not None
            else "   ?  "
        )
        detail = job.get("status_detail") or ""
        print(
            f"  {job['type']:<22}  {job['status']:<9}  {pct}  {detail}"
        )

    if all(j["status"] in TERMINAL for j in jobs):
        break
    time.sleep(POLL_INTERVAL)

print("\nAll jobs terminal.")

## 7. Inspect training results

`GET /latest/train/status/{session_id}/results` returns per-verse rows from every completed app, interleaved by `vref`. You can filter by `book` / `chapter` / `verse` and paginate with `page` + `page_size`.

Conceptually these results are `/predict` run over **the whole training text** — every app scores every verse you uploaded against the reference, so you get a free pre-scored pass over your translation as a side effect of training. That holds for `semantic-similarity`, `tfidf`, `word-alignment`, and `ngrams`.

The exception is **`agent-critique`**: running it on the entire submitted text would be prohibitively expensive (it's an LLM-driven critique loop). Training only fits the agent artifacts; it does not return per-verse agent results here. To get agent-critique scores for specific verses, call `/predict` (next section) with `apps=["agent-critique"]` and the verses you actually want critiqued.

The Nyanja corpus has the full Bible, so we filter to a single chapter (Luke 1) just to keep the preview small.

In [29]:
resp = check(requests.get(
    f"{BASE_URL}/{PREFIX}/train/status/{SESSION_ID}/results",
    params={"book": "LUK", "chapter": 1, "page": 1, "page_size": 10},
    headers=AUTH,
))
results = resp.json()
print(json.dumps(results, indent=2))

{
  "session_id": "45937f1e-5385-4421-b373-824d7175fe43",
  "training_jobs": [
    {
      "id": 73,
      "type": "semantic-similarity",
      "source_revision_id": 14558,
      "target_revision_id": 304,
      "source_version_id": 11191,
      "target_version_id": 296,
      "options": {
        "use_eflomal": true
      },
      "requested_time": "2026-05-01T04:38:19.359587",
      "owner_id": 207,
      "session_id": "45937f1e-5385-4421-b373-824d7175fe43",
      "assessment_id": 18078,
      "status": "finished",
      "status_detail": "Semantic similarity complete",
      "percent_complete": 100.0,
      "start_time": "2026-05-01T04:38:22.674061",
      "end_time": "2026-05-01T04:39:01.430470"
    },
    {
      "id": 74,
      "type": "tfidf",
      "source_revision_id": 14558,
      "target_revision_id": 304,
      "source_version_id": 11191,
      "target_version_id": 296,
      "options": {
        "use_eflomal": true
      },
      "requested_time": "2026-05-01T04:38:19.36827

## 8. Run `/predict` on a new verse pair

Once training is done, `/predict` scores arbitrary `(source_text, target_text)` pairs against the trained artifacts. The expectation is that each pair is either a **brand-new draft verse** that was not in the training data, or a **revised version of a verse** that was — so you can score quality after each translator edit without retraining.

- `pairs` — list of `{vref, source_text, target_text}`. **Scoring runs on the strings**: ngrams checks overlap with the trained ngram set, semantic-similarity compares text embeddings, and so on. `vref` is an optional convenience identifier — the API echoes it back alongside each pair's results so you can match scores to verses on the client side, but it does not drive scoring.
- `source_version_id` / `target_version_id` — the version pair the artifacts were trained under. After this migration, predict is **keyed on versions, not revisions**: training artifacts are shared across every revision within a version (so a fresh revision of an in-progress translation reuses the same trained models), but they don't leak across versions — two different versions in the same language are isolated, and you'll need to train each one separately.
- `apps` — optional list to fan out to a subset.

Below we score two pairs at LUK 1:1: the first uses the real Nyanja rendering paired with KJV English (a known-good match), the second pairs the same Nyanja text with a deliberately wrong English verse so you can see the score collapse.

In [ ]:
# Pull the actual Nyanja rendering of LUK 1:1 from the corpus we uploaded.
# LUK 1:1 is the 24,963rd canonical vref (1-indexed); see vref.txt at
# https://github.com/BibleNLP/ebible/blob/main/metadata/vref.txt
LUK_1_1_LINE = 24963
nya_luk_1_1 = REVISION_LINES[LUK_1_1_LINE - 1]
print(f"LUK 1:1 (Nyanja): {nya_luk_1_1!r}")

predict_payload = {
    "pairs": [
        {
            "vref": "LUK 1:1",
            "source_text": nya_luk_1_1,
            "target_text": (
                "Forasmuch as many have taken in hand to set forth in order "
                "a declaration of those things which are most surely "
                "believed among us,"
            ),
        },
        {
            "vref": "LUK 1:1",
            "source_text": nya_luk_1_1,
            # Adjusted target: same vref, but a clearly unrelated rendering
            "target_text": "And there was a great storm at sea, and the boat sank.",
        },
    ],
    "source_version_id": VERSION_ID,
    "target_version_id": REFERENCE_VERSION_ID,
    # "apps": ["semantic-similarity", "ngrams"],  # optional subset
}

resp = check(requests.post(
    f"{BASE_URL}/{PREFIX}/predict",
    json=predict_payload,
    headers=AUTH,
))
predictions = resp.json()
print(json.dumps(predictions, indent=2)[:3000])

## 9. (Optional) Cleanup

Demo runs accumulate. The cells below remove the revision and version you created. Run them only when you're done — they don't touch the reference (NIV) data.

Training jobs auto-clean themselves once they reach a terminal state, so there's nothing extra to delete on the training side.

In [15]:
# Uncomment to remove the demo revision and version.
check(requests.delete(
    f"{BASE_URL}/{PREFIX}/revision",
    params={"id": REVISION_ID},
    headers=AUTH,
))

check(requests.delete(
    f"{BASE_URL}/{PREFIX}/version",
    params={"id": VERSION_ID},
    headers=AUTH,
))

print("Cleaned up.")

Cleaned up.
